# Data Preprocessing Pipeline

> **Prerequisite:** Run `01_EDA.ipynb` first to generate `cleaned_*.csv` files.

## Objectives
1. Load EDA-cleaned datasets.
2. Engineer domain-specific features.
3. Handle class imbalance with SMOTETomek / ADASYN (training split only).
4. Build reusable `ColumnTransformer + Pipeline` preprocessors.
5. Perform stratified train/test splits.
6. Export `X_train`, `X_test`, `y_train`, `y_test` CSVs and `preprocessor.joblib`.

In [ ]:
# ── 0. Imports & Config ────────────────────────────────────────────────────
import time, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from imblearn.combine import SMOTETomek
from imblearn.over_sampling import ADASYN, SMOTE

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
TEST_SIZE = 0.20
os.makedirs('artifacts', exist_ok=True)
print('✅ Imports complete')

## Feature Engineering

In [ ]:
# ── 1. Load Cleaned Datasets ───────────────────────────────────────────────
cardio = pd.read_csv('cleaned_cardiovascular.csv')
diab   = pd.read_csv('cleaned_diabetes.csv')
heart  = pd.read_csv('cleaned_heart_disease.csv')
print(f'Cardiovascular : {cardio.shape}')
print(f'Diabetes       : {diab.shape}')
print(f'Heart Disease  : {heart.shape}')

In [ ]:
# ── 2. Feature Engineering ─────────────────────────────────────────────────

def engineer_cardio(df):
    df = df.copy()
    if 'age_years' not in df.columns:
        df['age_years'] = (df['age'] / 365.25).round(1)
    # BMI
    df['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)
    # Pulse pressure (systolic - diastolic) — clinical indicator
    df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']
    # Hypertension flag (Stage 2: systolic ≥ 140)
    df['hypertension'] = ((df['ap_hi'] >= 140) | (df['ap_lo'] >= 90)).astype(int)
    # Overweight flag
    df['overweight'] = (df['bmi'] >= 25).astype(int)
    # Drop raw age (days) and original height/weight — BMI captures them
    return df.drop(columns=['age'], errors='ignore')

def engineer_brfss(df):
    df = df.copy()
    # Composite lifestyle score (more positive = healthier)
    df['lifestyle_score'] = (df['PhysActivity'] + df['Fruits'] + df['Veggies']
                              - df['Smoker'] - df['HvyAlcoholConsump'])
    # Comorbidity count
    df['comorbidity_count'] = (df['HighBP'] + df['HighChol'] + df['Stroke']).astype(int)
    # Health burden (poor general + physical + mental)
    df['health_burden'] = df['GenHlth'] + (df['PhysHlth'] > 14).astype(int) + (df['MentHlth'] > 14).astype(int)
    return df

cardio = engineer_cardio(cardio)
diab   = engineer_brfss(diab)
heart  = engineer_brfss(heart)

print('✅ Feature engineering complete')
print(f'Cardiovascular new cols: {[c for c in cardio.columns if c not in pd.read_csv("cleaned_cardiovascular.csv").columns]}')
print(f'BRFSS new cols: {["lifestyle_score", "comorbidity_count", "health_burden"]}')

## Handling Missing Values

No missing values were found in EDA. The `SimpleImputer` is included in the pipeline as a safeguard for production robustness (e.g., when the pipeline receives partial real-world data).

In [ ]:
# ── 3. Define Feature Schemas ──────────────────────────────────────────────

CARDIO_TARGET = 'cardio'
DIAB_TARGET   = 'Diabetes_binary'
HEART_TARGET  = 'HeartDiseaseorAttack'

# Cardiovascular
cardio_num = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi', 'pulse_pressure']
cardio_cat = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'hypertension', 'overweight']

# BRFSS
brfss_num = ['BMI', 'MentHlth', 'PhysHlth', 'lifestyle_score', 'comorbidity_count', 'health_burden']
brfss_cat = ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
              'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump',
              'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex',
              'GenHlth', 'Age', 'Education', 'Income']

diab_cat  = [c for c in brfss_cat if c != 'HeartDiseaseorAttack' and c in diab.columns]
heart_cat = [c for c in brfss_cat if c != 'HeartDiseaseorAttack' and c in heart.columns]
# Note: for diabetes dataset, HeartDiseaseorAttack is a feature (not target)
diab_cat_all = [c for c in brfss_cat if c in diab.columns]

print('✅ Feature schemas defined')

In [ ]:
# ── 4. Preprocessing Pipeline Factory ─────────────────────────────────────

def build_preprocessor(num_cols, cat_cols):
    """Build a ColumnTransformer that scales numeric and ordinal-encodes categorical features."""
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_pipe, num_cols),
        ('cat', categorical_pipe, cat_cols)
    ], remainder='drop')
    return preprocessor

print('✅ Preprocessor factory ready')

## Data Imbalance Handling

Class imbalance is a critical concern in healthcare datasets:

| Dataset | Minority class % | Strategy |
|---------|-----------------|----------|
| Cardiovascular | ~50% | Balanced — no resampling needed |
| Diabetes | ~14% | **SMOTETomek** (oversamples minority + removes Tomek link noise) |
| Heart Disease | ~9% | **ADASYN** (adaptive density-based synthesis — focuses on hard-to-classify samples) |

⚠️ Resampling is applied **only to training data** to prevent data leakage.

In [ ]:
# ── 5. Imbalance Analysis ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (df, target, title) in zip(axes, [
    (cardio, CARDIO_TARGET, 'Cardiovascular'),
    (diab,   DIAB_TARGET,   'Diabetes'),
    (heart,  HEART_TARGET,  'Heart Disease'),
]):
    vc = df[target].value_counts().sort_index()
    bars = ax.bar(vc.index.astype(str), vc.values,
                  color=['#4CAF50', '#E53935'], edgecolor='white')
    ax.set_title(f'{title}\nImbalance Ratio: {vc.max()/vc.min():.1f}:1', fontweight='bold')
    for bar, v in zip(bars, vc.values):
        pct = v / vc.sum() * 100
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{v:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.suptitle('Class Distribution Before Resampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance_before.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 6. Train-Test Split (Stratified) ──────────────────────────────────────

def split_dataset(df, target, num_cols, cat_cols, dataset_name):
    """Stratified split, fit preprocessor on train, apply SMOTETomek/ADASYN to train only."""
    t0 = time.time()
    feature_cols = [c for c in num_cols + cat_cols if c in df.columns]
    X = df[feature_cols]
    y = df[target].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    # Fit preprocessor on training data only
    preprocessor = build_preprocessor(
        [c for c in num_cols if c in feature_cols],
        [c for c in cat_cols if c in feature_cols]
    )
    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc  = preprocessor.transform(X_test)

    print(f'\n📊 {dataset_name}')
    print(f'   Train: {X_train_proc.shape}, Test: {X_test_proc.shape}')
    print(f'   Train class dist: {pd.Series(y_train).value_counts().to_dict()}')

    return X_train_proc, X_test_proc, y_train, y_test, preprocessor

# --- Cardiovascular (balanced, no resampling needed) ---
X_tr_c, X_te_c, y_tr_c, y_te_c, prep_c = split_dataset(
    cardio, CARDIO_TARGET, cardio_num, cardio_cat, 'Cardiovascular'
)

# --- Diabetes (SMOTETomek) ---
X_tr_d, X_te_d, y_tr_d, y_te_d, prep_d = split_dataset(
    diab, DIAB_TARGET, brfss_num, diab_cat_all, 'Diabetes'
)

# --- Heart Disease (ADASYN) ---
X_tr_h, X_te_h, y_tr_h, y_te_h, prep_h = split_dataset(
    heart, HEART_TARGET, brfss_num, heart_cat, 'Heart Disease'
)

print('\n✅ Splits complete')

In [ ]:
# ── 7. Apply Resampling (training only) ────────────────────────────────────

def apply_resampling(X_train, y_train, strategy, dataset_name):
    """Apply resampling and report class counts before/after."""
    t0 = time.time()
    before = pd.Series(y_train).value_counts().to_dict()

    if strategy == 'smotetomek':
        sampler = SMOTETomek(random_state=RANDOM_STATE)
    elif strategy == 'adasyn':
        sampler = ADASYN(random_state=RANDOM_STATE)
    elif strategy == 'smote':
        sampler = SMOTE(random_state=RANDOM_STATE)
    else:
        return X_train, y_train  # no resampling

    X_res, y_res = sampler.fit_resample(X_train, y_train)
    after = pd.Series(y_res).value_counts().to_dict()
    elapsed = time.time() - t0
    print(f'  {dataset_name} [{strategy}] — Before: {before} → After: {after} | {elapsed:.1f}s')
    return X_res, y_res

print('Applying resampling strategies...')
# Cardiovascular: balanced, skip
X_tr_c_res, y_tr_c_res = X_tr_c, y_tr_c

# Diabetes: SMOTETomek
X_tr_d_res, y_tr_d_res = apply_resampling(X_tr_d, y_tr_d, 'smotetomek', 'Diabetes')

# Heart Disease: ADASYN (better for hard-to-classify boundary cases)
X_tr_h_res, y_tr_h_res = apply_resampling(X_tr_h, y_tr_h, 'adasyn', 'Heart Disease')

print('\n✅ Resampling complete')

In [ ]:
# ── 8. Visualise Class Distribution After Resampling ──────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

pairs = [
    (y_tr_c, y_tr_c_res, 'Cardiovascular\n(no resampling)'),
    (y_tr_d, y_tr_d_res, 'Diabetes\n(SMOTETomek)'),
    (y_tr_h, y_tr_h_res, 'Heart Disease\n(ADASYN)'),
]

for col, (before, after, title) in enumerate(pairs):
    for row, (y_data, label) in enumerate([(before, 'Before'), (after, 'After')]):
        vc = pd.Series(y_data).value_counts().sort_index()
        axes[row, col].bar(vc.index.astype(str), vc.values,
                           color=['#4CAF50','#E53935'], edgecolor='white')
        axes[row, col].set_title(f'{title}\n{label} Resampling', fontweight='bold', fontsize=9)
        for i, v in enumerate(vc.values):
            axes[row, col].text(i, v * 1.01, f'{v:,}', ha='center', fontsize=8)

plt.suptitle('Class Distribution Before vs After Resampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance_after.png', bbox_inches='tight', dpi=150)
plt.show()

## Artifact Saving

In [ ]:
# ── 9. Save Splits and Preprocessors ──────────────────────────────────────
import os

def save_splits(prefix, X_tr, X_te, y_tr, y_te, preprocessor):
    pd.DataFrame(X_tr).to_csv(f'{prefix}_X_train.csv', index=False)
    pd.DataFrame(X_te).to_csv(f'{prefix}_X_test.csv',  index=False)
    pd.Series(y_tr, name='target').to_csv(f'{prefix}_y_train.csv', index=False)
    pd.Series(y_te, name='target').to_csv(f'{prefix}_y_test.csv',  index=False)
    joblib.dump(preprocessor, f'artifacts/{prefix}_preprocessor.joblib')
    print(f'  ✅ {prefix}: splits + preprocessor saved')

save_splits('cardio', X_tr_c_res, X_te_c, y_tr_c_res, y_te_c, prep_c)
save_splits('diabetes', X_tr_d_res, X_te_d, y_tr_d_res, y_te_d, prep_d)
save_splits('heart', X_tr_h_res, X_te_h, y_tr_h_res, y_te_h, prep_h)

print('\n📁 Artifact directory contents:')
for f in sorted(os.listdir('artifacts')):
    print(f'   artifacts/{f}')

## Final Pipeline Summary

```
Raw CSV
  └─ engineer_cardio() / engineer_brfss()      ← domain feature engineering
       └─ train_test_split(stratify=y)           ← stratified 80/20 split
            ├─ X_train → ColumnTransformer.fit_transform()
            │     ├─ numeric: median impute → StandardScaler
            │     └─ categorical: mode impute → OrdinalEncoder
            │         └─ SMOTETomek / ADASYN (training only)
            └─ X_test  → ColumnTransformer.transform()  (no leakage)
```

All preprocessors saved as `.joblib` — load in `03_Modeling.ipynb`.